# Setup

In [ ]:
import duckdb
import logging
from splink import DuckDBAPI, Linker, SettingsCreator, block_on
import splink.comparison_library as cl
import splink.comparison_level_library as cll
import splink.blocking_rule_library as brl

from splink.blocking_analysis import (
    count_comparisons_from_blocking_rules,
    chart_comparisons_from_blocking_rules,
    n_largest_blocks,
)

con = duckdb.connect()

df_l = con.read_parquet("./perturbed_data_l.parquet")
df_r = con.read_parquet("./perturbed_data_r.parquet")

# preprocessing to get fname rarity flag

df_concat = con.sql(
    """
    select *, 'df_l' as source_dataset
    from df_l
    
    union all
    
    select *, 'df_r' as source_dataset
    from df_r
    """
)

fn_counts = con.sql(
    """
    select
        fname,
        case
            when count(*) < 800 then 1
            else NULL
        end as fn_rare_flag
    from df_concat
    where fname is not null
    group by fname
    """
)

df_l = con.sql(
    """
    select *
    from df_l
    left join fn_counts
    using (fname)
    """
)

df_r = con.sql(
    """
    select *
    from df_r
    left join fn_counts
    using (fname)
    """
)

db_api = DuckDBAPI(connection=con)

df_l = db_api.register(df_l, table_name="df_l")
df_r = db_api.register(df_r, table_name="df_r")

In [2]:
# have a look at the data
con.sql("select * from df_l")

# Blocking

In [ ]:
brs = [
    block_on("substring(fname, 1, 2)", "substring(sname, 1, 1)"),
    block_on("substring(fname, 1, 2)", "year(dob)"),
    block_on("substring(fname, 1, 2)", "sa4", arrays_to_explode=["sa4"]),
    block_on("substring(sname, 1, 2)", "substring(fname, 1, 1)"),
    block_on("substring(sname, 1, 2)", "year(dob)"),
    block_on("substring(sname, 1, 2)", "sa4", arrays_to_explode=["sa4"]),
    block_on("year(dob)", "day(dob)"),
    block_on("address", arrays_to_explode=["address"]),
    block_on("fname", "fn_rare_flag"),
]

chart_comparisons_from_blocking_rules(
    splink_dataframe_or_dataframes=[df_l, df_r],
    blocking_rules=brs,
    link_type='link_and_dedupe',
    record_sample_proportion=0.1,
)

In [ ]:
n_largest_blocks(
    splink_dataframe_or_dataframes=[df_l, df_r],
    blocking_rule=block_on("fname", "fn_rare_flag"),
    link_type='link_and_dedupe',
    n_largest=10,
).as_duckdbpyrelation()

# Evaluation

In [ ]:
# recall of blocking rules calculated using person_id

# make each blocking rule stricter to only include true matches
brs_and_person_id = [
    brl.And(br, block_on("person_id"), arrays_to_explode=br.arrays_to_explode)
    for br in brs
]

settings = SettingsCreator(
    link_type="link_and_dedupe",
    comparisons=[],
    blocking_rules_to_generate_predictions=brs_and_person_id,
    retain_intermediate_calculation_columns=False,
    retain_matching_columns=False,
)

linker = Linker([df_l, df_r], settings, log_level=20)

# deterministic link means we take every pair returned by the blocking rules as true links
df_pairs = linker.inference.deterministic_link()

res = con.sql(
    f"""
    select
        count(*) as count
    from {df_pairs.physical_name}
    """
)

found_matches_count = res.fetchall()[0][0]

linker.table_management.delete_tables_created_by_splink_from_db()

In [ ]:
settings = SettingsCreator(
    link_type="link_and_dedupe",
    comparisons=[],
    blocking_rules_to_generate_predictions=[block_on("person_id")],
    retain_intermediate_calculation_columns=False,
    retain_matching_columns=False,
)

linker = Linker([df_l, df_r], settings, log_level=20)

df_pairs = linker.inference.deterministic_link()

res = con.sql(
    f"""
    select
        count(*) as count
    from {df_pairs.physical_name}
    """
)

true_matches_count = res.fetchall()[0][0]

linker.table_management.delete_tables_created_by_splink_from_db()

print(f"Found matches = {found_matches_count}")
print(f"True matches = {true_matches_count}")
print(f"Proportion of true matches found by blocking rules: {found_matches_count / true_matches_count}")

# Advanced blocking - Locality Sensitive Hashing

Locality Sensitive Hashing (LSH) is a technique for randomly placing records into blocks so that records that are more similar are more likely to be placed in the same block. This is a more advanced technique to implement, but it is a more general approach to blocking than selecting blocking rules by hand.

The idea behind LSH is to split each record into a collection of q-grams (overlapping strings of length q). The similarity of two records is measured using the Jaccard similarity of the two sets of q-grams, which is the size of their intersection divided by the size of their union. For each record, some number of q-grams are selected using min-hashes (see resources for more information). These are divided up into b bands of r q-grams. Each record is placed into b blocks, and each block of those blocks is the set of records that agree on those r q-grams. 
The numbers b and r are the two main parameters for LSH. Increasing b means that records have a higher chance to be placed in the same block, and increasing r means that two records have to be more similar to be placed into the same block. 

LSH has some of the same problems as selecting blocking rules by hand. You can still get some high frequency values that create very large blocks, which generates a lot of comparisons. LSH can be combined with techniques that filter out high-frequency blocks (or rank based filtering as used in the first resource).

LSH is not implemented in Splink.

**Resources**

- [Locality Sensitive Hashing with Temporal and Spatial Constraints for
Efficient Population Record Linkage](https://dl.acm.org/doi/pdf/10.1145/3511808.3557631)
- [UNIQUE ENTITY ESTIMATION WITH APPLICATION TO THE SYRIAN CONFLICT](https://www.jstor.org/stable/26542562)
- [Blocking and Filtering Techniques for Entity Resolution:
A Survey](https://dl.acm.org/doi/pdf/10.1145/3377455)



In [ ]:
# very rough implementation to demonstrate LSH
def create_q_grams(connection, tbl, blocking_cols, id_cols, q):
    select_blocking_cols = ", ".join(blocking_cols)

    tbl_with_blocking_string = connection.sql(
        f"""
        select 
            {','.join(id_cols)},
            concat_ws('_', {select_blocking_cols}) as blocking_string
        from tbl
        """
    )

    tbl_with_qgrams = connection.sql(
        f"""
        select
            {','.join(id_cols)},
            list_transform(
                range(0, length(blocking_string) - {q} + 1),
                lambda i : blocking_string[i+1 : i+{q}]
            ) as qgrams
        from tbl_with_blocking_string
        """
    )

    return tbl_with_qgrams

def create_lsh_blocks(connection, tbl, blocking_cols, id_cols, q, b, r):
    """
    Inputs:
        - connection: the DuckDB connection object
        - tbl: the input table as a DuckDB relation object (or a pandas dataframe)
        - blocking_cols: a list of column names to use in the LSH steps
        - id_column: the name of the column that uniquely identifies the units in the input table
        - q: the number q for which to create q-grams (e.g. 3-grams would be triples of letters 'ABC' or 'XYZ')
        - b: the number of bands for LSH
        - r: the number of rows for LSH

    Returns:
        A DuckDB relation with the id_column and min_hash columns.
    """
    qgrams = create_q_grams(
        connection=connection,
        tbl=tbl,
        blocking_cols=blocking_cols,
        id_cols=id_cols,
        q=q,
    )

    min_hash_exprs = [
        f"list_min(list_transform(qgrams, lambda x : hash(x, {i}))) as min_hash_{i}"
        for i in range(b * r)
    ]

    select_hashes = ", ".join(min_hash_exprs)

    min_hashes = connection.sql(
        f"""
        select
            {','.join(id_cols)},
            {select_hashes}
        from qgrams
        """
    )

    return min_hashes

In [ ]:
# check the q-grams look reasonable
#
# Note: if we just use dob it includes all the datetime components which creates a lot of spurious similarity between
# records (from hours/minutes/seconds)

qgrams = create_q_grams(
    con, 
    df_concat, 
    blocking_cols=["fname", "sname", "strftime(dob, '%y%m%d')", "address[1]", "address[2]", "sa4[1]", "sa4[2]"],
    id_cols=["unique_id", "source_dataset"],
    q=3
)
qgrams

In [ ]:
con.sql("drop table if exists blocks")
B = 40
R = 3

blocks = create_lsh_blocks(
    con,
    df_concat,
    blocking_cols=["fname", "sname", "strftime(dob, '%y%m%d')", "address[1]", "address[2]", "sa4[1]", "sa4[2]"],
    id_cols=["unique_id", "source_dataset"],
    q=3,
    b=B,
    r=R,
)

# create the blocks table to avoid recomputing
blocks.create("blocks")

In [ ]:
# note one disadvantage here: this table has B * R new columns, which means more data has to be processed overall
df_lsh = con.sql(
    """
    select *
    from df_concat
    join blocks
    using (unique_id, source_dataset)
    """
)

df_lsh = db_api.register(df_lsh)

# the blocking rules for LSH are groups of R min_hashes, repeated B times
lsh_brs = [
    block_on(*[f"min_hash_{i}" for i in range(R * j, R * (j + 1))])
    for j in range(B)
]

chart_comparisons_from_blocking_rules(
    [df_lsh],
    blocking_rules=lsh_brs,
    link_type="link_and_dedupe",
    record_sample_proportion=0.2,
)

In [ ]:
# recall of blocking rules calculated using person_id
brs_and_person = [
    brl.And(br, block_on("person_id"), arrays_to_explode=br.arrays_to_explode)
    for br in lsh_brs
]

settings = SettingsCreator(
    link_type="link_and_dedupe",
    comparisons=[],
    blocking_rules_to_generate_predictions=brs_and_person,
    retain_intermediate_calculation_columns=False,
    retain_matching_columns=False,
)

linker = Linker([df_lsh], settings)

df_pairs = linker.inference.deterministic_link()

res = con.sql(
    f"""
    select
        count(*) as count
    from {df_pairs.physical_name}
    """
)

display(res)

linker.table_management.delete_tables_created_by_splink_from_db()